# Kalenjin ASR: Advanced Modular Preprocessing Pipeline

## Speech Recognition Data Preprocessing Framework

**Author**: Research Team  
**Dataset**: Mozilla Common Voice v24.0 - Kalenjin (kln)  
**Purpose**: Comprehensive modular preprocessing for ASR model training  
**Target Models**: Whisper, Wav2Vec2, MMS, WavLM

# Abstract

This notebook implements a state-of-the-art modular preprocessing pipeline for Kalenjin ASR, incorporating advanced signal processing techniques, linguistic normalization, and quality assessment protocols. The pipeline is designed for reproducibility and scalability across multiple pretrained model architectures.

### Key Features:
- **Modular Architecture**: Independent, reusable components
- **Multi-Model Support**: Optimized for Whisper, Wav2Vec2, MMS, WavLM
- **Advanced Audio Processing**: VAD, spectral analysis, quality metrics
- **Linguistic Normalization**: Kalenjin-specific text processing
- **Quality Assurance**: Comprehensive validation and filtering
- **Reproducibility**: Deterministic processing with versioning

## Table of Contents
1. [Environment Setup & Dependencies](#section1)
2. [Configuration Management](#section2)
3. [Audio Processing Module](#section3)
4. [Text Normalization Module](#section4)
5. [Quality Assessment Module](#section5)
6. [Dataset Structuring Module](#section6)
7. [Pipeline Orchestration](#section7)
8. [Validation & Quality Control](#section8)
9. [Export & Model Preparation](#section9)
10. [Performance Analysis](#section10)

## 1. Environment Setup & Dependencies <a id='section1'></a>

### Theoretical Foundation
Modern ASR preprocessing requires careful consideration of:
- **Nyquist-Shannon Sampling Theorem**: Proper resampling to avoid aliasing
- **Psychoacoustic Principles**: Perceptually-motivated filtering
- **Information Theory**: Optimal text normalization for vocabulary reduction
- **Statistical Signal Processing**: Robust quality metrics and outlier detection

# Core scientific computing
import numpy as np
import pandas as pd
from pathlib import Path
import json
import yaml
from typing import Dict, List, Tuple, Optional, Union, Any
from dataclasses import dataclass, asdict
from datetime import datetime
import logging
import warnings
warnings.filterwarnings('ignore')

# Audio processing & signal analysis
import librosa
import librosa.display
import soundfile as sf
import scipy.signal
from scipy.stats import zscore, iqr
import webrtcvad
import pyannote.audio
from pyannote.audio import Pipeline

# Text processing & linguistics
import re
import unicodedata
from collections import Counter, defaultdict
import string
from textdistance import levenshtein

# Machine learning & data science
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import datasets
from datasets import Dataset, DatasetDict, Audio

# Visualization & analysis
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Parallel processing
from multiprocessing import Pool, cpu_count
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from tqdm.auto import tqdm
tqdm.pandas()

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('preprocessing.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('✓ All dependencies loaded successfully')
print(f'✓ Available CPU cores: {cpu_count()}')
print(f'✓ Librosa version: {librosa.__version__}')
print(f'✓ Datasets version: {datasets.__version__}')

In [1]:
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CORE DEPENDENCIES
# ============================================================================
import numpy as np
import pandas as pd
from pathlib import Path
import json
from typing import Dict, List, Tuple, Optional, Union, Any
from dataclasses import dataclass, asdict
from datetime import datetime
import logging

# Audio processing
import librosa
import librosa.display
import soundfile as sf
import scipy.signal
from scipy.stats import zscore, iqr

# Text processing
import re
import unicodedata
from collections import Counter, defaultdict
import string

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import datasets
from datasets import Dataset, DatasetDict, Audio

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Parallel processing
from multiprocessing import Pool, cpu_count
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from tqdm.auto import tqdm
tqdm.pandas()

# ============================================================================
# OPTIONAL DEPENDENCIES (with fallbacks)
# ============================================================================
try:
    import webrtcvad
    WEBRTC_AVAILABLE = True
except:
    WEBRTC_AVAILABLE = False

try:
    import yaml
    YAML_AVAILABLE = True
except:
    YAML_AVAILABLE = False

try:
    from textdistance import levenshtein
    TEXTDISTANCE_AVAILABLE = True
except:
    TEXTDISTANCE_AVAILABLE = False

try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except:
    PYDUB_AVAILABLE = False

# ============================================================================
# CONFIGURATION
# ============================================================================
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# ============================================================================
# FALLBACK IMPLEMENTATIONS
# ============================================================================
class SimpleVAD:
    """Librosa-based VAD fallback."""
    def __init__(self, aggressiveness=2):
        self.top_db = [15, 20, 25, 30][aggressiveness]
    
    def trim_silence(self, audio, sr=16000):
        trimmed, _ = librosa.effects.trim(audio, top_db=self.top_db)
        return trimmed

if not WEBRTC_AVAILABLE:
    webrtcvad = type('webrtcvad', (), {'Vad': SimpleVAD})

# ============================================================================
# ENVIRONMENT SUMMARY
# ============================================================================
print('=' * 70)
print('KALENJIN ASR PREPROCESSING PIPELINE'.center(70))
print('=' * 70)
print(f'\n✓ NumPy:      {np.__version__}')
print(f'✓ Pandas:     {pd.__version__}')
print(f'✓ Librosa:    {librosa.__version__}')
print(f'✓ Datasets:   {datasets.__version__}')
print(f'✓ Matplotlib: {plt.matplotlib.__version__}')
print(f'✓ CPU Cores:  {cpu_count()}')
print(f'\nVAD Method:   {"WebRTC" if WEBRTC_AVAILABLE else "Librosa (fallback)"}')
print('=' * 70)
print('✓ Ready for preprocessing!\n')


                 KALENJIN ASR PREPROCESSING PIPELINE                  

✓ NumPy:      2.2.6
✓ Pandas:     2.3.3
✓ Librosa:    0.11.0
✓ Datasets:   4.5.0
✓ Matplotlib: 3.10.8
✓ CPU Cores:  16

VAD Method:   Librosa (fallback)
✓ Ready for preprocessing!



---
## 2. Configuration Management <a id='section2'></a>

### Design Philosophy
Centralized configuration management ensures reproducibility and enables systematic hyperparameter exploration. The configuration system supports:
- **Model-specific parameters**: Optimized for different ASR architectures
- **Linguistic constraints**: Kalenjin-specific processing rules
- **Quality thresholds**: Evidence-based filtering criteria
- **Processing options**: Scalable parallel processing settings

In [2]:
@dataclass
class AudioConfig:
    """Audio processing configuration with model-specific optimizations."""
    
    # Core audio parameters
    target_sr: int = 16000  # Standard for most ASR models
    mono: bool = True
    normalize: bool = True
    
    # Duration filtering (evidence-based thresholds)
    min_duration: float = 0.5  # Remove breath sounds, clicks
    max_duration: float = 20.0  # Memory constraints, alignment issues
    
    # Voice Activity Detection
    vad_mode: int = 3  # Most aggressive VAD (0-3)
    vad_frame_duration: int = 30  # ms
    silence_threshold: float = 0.01  # RMS threshold
    
    # Quality metrics thresholds
    min_snr_db: float = 10.0  # Signal-to-noise ratio
    max_zero_crossing_rate: float = 0.3  # Detect corrupted audio
    min_spectral_centroid: float = 200.0  # Hz
    max_spectral_centroid: float = 8000.0  # Hz
    
    # Model-specific configurations
    whisper_compatible: bool = True
    wav2vec2_compatible: bool = True
    mms_compatible: bool = True

@dataclass
class TextConfig:
    """Text normalization configuration for Kalenjin language."""
    
    # Case normalization
    lowercase: bool = True
    
    # Character filtering
    allowed_chars: str = 'abcdefghijklmnopqrstuvwxyz '
    remove_punctuation: bool = True
    preserve_apostrophes: bool = True  # Important for Kalenjin "ng'"
    
    # Kalenjin-specific rules
    normalize_tones: bool = True
    standardize_orthography: bool = True
    
    # Quality thresholds
    min_words: int = 1
    max_words: int = 50
    max_char_repetition: int = 3
    
    # Vocabulary constraints
    min_word_freq: int = 2  # Remove hapax legomena
    max_oov_ratio: float = 0.1  # Out-of-vocabulary threshold

@dataclass
class ProcessingConfig:
    """Processing and system configuration."""
    
    # Parallel processing
    n_jobs: int = min(cpu_count() - 1, 8)
    batch_size: int = 1000
    
    # Memory management
    max_memory_gb: float = 8.0
    cache_processed: bool = True
    
    # Reproducibility
    random_seed: int = 42
    deterministic: bool = True
    
    # Output configuration
    save_intermediate: bool = True
    export_formats: List[str] = None
    
    def __post_init__(self):
        if self.export_formats is None:
            self.export_formats = ['huggingface', 'json', 'csv']

@dataclass
class PreprocessingConfig:
    """Master configuration combining all preprocessing parameters."""
    
    audio: AudioConfig
    text: TextConfig
    processing: ProcessingConfig
    
    # Metadata
    version: str = "1.0.0"
    created_at: str = None
    description: str = "Kalenjin ASR preprocessing pipeline"
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now().isoformat()
    
    def save(self, path: Union[str, Path]) -> None:
        """Save configuration to YAML file."""
        with open(path, 'w') as f:
            yaml.dump(asdict(self), f, default_flow_style=False)
    
    @classmethod
    def load(cls, path: Union[str, Path]) -> 'PreprocessingConfig':
        """Load configuration from YAML file."""
        with open(path, 'r') as f:
            data = yaml.safe_load(f)
        
        return cls(
            audio=AudioConfig(**data['audio']),
            text=TextConfig(**data['text']),
            processing=ProcessingConfig(**data['processing']),
            version=data.get('version', '1.0.0'),
            created_at=data.get('created_at'),
            description=data.get('description', '')
        )

# Initialize default configuration
config = PreprocessingConfig(
    audio=AudioConfig(),
    text=TextConfig(),
    processing=ProcessingConfig()
)

# Set random seeds for reproducibility
np.random.seed(config.processing.random_seed)

print('✓ Configuration system initialized')
print(f'✓ Processing with {config.processing.n_jobs} cores')
print(f'✓ Target sample rate: {config.audio.target_sr} Hz')
print(f'✓ Duration range: {config.audio.min_duration}-{config.audio.max_duration}s')

✓ Configuration system initialized
✓ Processing with 8 cores
✓ Target sample rate: 16000 Hz
✓ Duration range: 0.5-20.0s


---
## 3. Audio Processing Module <a id='section3'></a>

### Signal Processing Theory

The audio processing module implements state-of-the-art techniques:

1. **Resampling**: Uses Kaiser window for anti-aliasing (β=5.0)
2. **Voice Activity Detection**: WebRTC VAD + energy-based detection
3. **Quality Assessment**: Multi-dimensional audio quality metrics
4. **Normalization**: Peak normalization with dynamic range preservation

### Mathematical Foundations

- **SNR Calculation**: $SNR_{dB} = 10 \log_{10}\left(\frac{P_{signal}}{P_{noise}}\right)$
- **Spectral Centroid**: $C = \frac{\sum_{k=0}^{N-1} k \cdot |X(k)|}{\sum_{k=0}^{N-1} |X(k)|}$
- **Zero Crossing Rate**: $ZCR = \frac{1}{2N} \sum_{n=1}^{N-1} |sgn(x[n]) - sgn(x[n-1])|$

In [3]:
class AudioProcessor:
    """Advanced audio processing module for ASR preprocessing."""
    
    def __init__(self, config: AudioConfig):
        self.config = config
        self.vad = webrtcvad.Vad(config.vad_mode)
        self.stats = defaultdict(list)
        
        logger.info(f"AudioProcessor initialized with SR={config.target_sr}Hz")
    
    def load_audio(self, file_path: Union[str, Path]) -> Tuple[np.ndarray, int]:
        """Load audio file with error handling and format validation."""
        try:
            # Load with librosa for robust format support
            audio, sr = librosa.load(
                file_path, 
                sr=None,  # Preserve original sample rate initially
                mono=self.config.mono,
                dtype=np.float32
            )
            
            if len(audio) == 0:
                raise ValueError("Empty audio file")
            
            return audio, sr
            
        except Exception as e:
            logger.error(f"Failed to load {file_path}: {e}")
            return None, None
    
    def resample_audio(self, audio: np.ndarray, orig_sr: int) -> np.ndarray:
        """High-quality resampling with anti-aliasing."""
        if orig_sr == self.config.target_sr:
            return audio
        
        # Use librosa's high-quality resampling
        resampled = librosa.resample(
            audio, 
            orig_sr=orig_sr, 
            target_sr=self.config.target_sr,
            res_type='kaiser_fast'  # Good balance of quality/speed
        )
        
        return resampled
    
    # def apply_vad(self, audio: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
    #     """Advanced Voice Activity Detection with multiple methods."""
    #     sr = self.config.target_sr
        
    #     # Method 1: WebRTC VAD
    #     frame_duration = self.config.vad_frame_duration  # ms
    #     frame_length = int(sr * frame_duration / 1000)
        
    #     # Convert to 16-bit PCM for WebRTC VAD
    #     audio_int16 = (audio * 32767).astype(np.int16)
        
    #     vad_frames = []
    #     for i in range(0, len(audio_int16) - frame_length, frame_length):
    #         frame = audio_int16[i:i + frame_length]
    #         if len(frame) == frame_length:
    #             is_speech = self.vad.is_speech(frame.tobytes(), sr)
    #             vad_frames.append(is_speech)
        
    #     # Method 2: Energy-based VAD
    #     rms_energy = librosa.feature.rms(y=audio, frame_length=frame_length)[0]
    #     energy_threshold = np.percentile(rms_energy, 30)  # Adaptive threshold
    #     energy_vad = rms_energy > max(energy_threshold, self.config.silence_threshold)
        
    #     # Combine VAD methods
    #     if len(vad_frames) > 0:
    #         # Align frame counts
    #         min_frames = min(len(vad_frames), len(energy_vad))
    #         combined_vad = np.logical_or(
    #             vad_frames[:min_frames], 
    #             energy_vad[:min_frames]
    #         )
    #     else:
    #         combined_vad = energy_vad
        
    #     # Apply VAD to audio
    #     if np.any(combined_vad):
    #         # Find speech segments
    #         speech_frames = np.where(combined_vad)[0]
    #         start_frame = speech_frames[0] * frame_length
    #         end_frame = min((speech_frames[-1] + 1) * frame_length, len(audio))
            
    #         trimmed_audio = audio[start_frame:end_frame]
    #     else:
    #         # No speech detected, return original (might be very quiet speech)
    #         trimmed_audio = audio
        
    #     # VAD statistics
    #     vad_stats = {
    #         'speech_ratio': np.mean(combined_vad) if len(combined_vad) > 0 else 0.0,
    #         'original_length': len(audio) / sr,
    #         'trimmed_length': len(trimmed_audio) / sr,
    #         'reduction_ratio': 1 - (len(trimmed_audio) / len(audio))
    #     }
        
    #     return trimmed_audio, vad_stats
    
    def apply_vad(self, audio: np.ndarray) -> Tuple[np.ndarray, Dict[str, float]]:
        """Voice Activity Detection using librosa."""
        sr = self.config.target_sr
        
        # Use librosa trim (simple and effective)
        trimmed, _ = librosa.effects.trim(audio, top_db=20)
        
        # VAD statistics
        vad_stats = {
            'speech_ratio': len(trimmed) / len(audio) if len(audio) > 0 else 0.0,
            'original_length': len(audio) / sr,
            'trimmed_length': len(trimmed) / sr,
            'reduction_ratio': 1 - (len(trimmed) / len(audio)) if len(audio) > 0 else 0.0
        }
        
        return trimmed, vad_stats

    
    def calculate_quality_metrics(self, audio: np.ndarray) -> Dict[str, float]:
        """Comprehensive audio quality assessment."""
        sr = self.config.target_sr
        
        # Basic statistics
        duration = len(audio) / sr
        rms_energy = np.sqrt(np.mean(audio**2))
        peak_amplitude = np.max(np.abs(audio))
        
        # Signal-to-Noise Ratio estimation
        # Use quietest 10% as noise estimate
        sorted_energy = np.sort(np.abs(audio))
        noise_floor = np.mean(sorted_energy[:int(0.1 * len(sorted_energy))])
        signal_power = rms_energy**2
        noise_power = noise_floor**2
        snr_db = 10 * np.log10(signal_power / (noise_power + 1e-10))
        
        # Spectral features
        stft = librosa.stft(audio)
        magnitude = np.abs(stft)
        
        # Spectral centroid (brightness)
        spectral_centroids = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
        mean_spectral_centroid = np.mean(spectral_centroids)
        
        # Zero crossing rate (indicates noisiness)
        zcr = librosa.feature.zero_crossing_rate(audio)[0]
        mean_zcr = np.mean(zcr)
        
        # Spectral rolloff (frequency content)
        spectral_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)[0]
        mean_rolloff = np.mean(spectral_rolloff)
        
        # Dynamic range
        dynamic_range_db = 20 * np.log10(peak_amplitude / (rms_energy + 1e-10))
        
        # Clipping detection
        clipping_ratio = np.sum(np.abs(audio) > 0.99) / len(audio)
        
        return {
            'duration': duration,
            'rms_energy': rms_energy,
            'peak_amplitude': peak_amplitude,
            'snr_db': snr_db,
            'spectral_centroid': mean_spectral_centroid,
            'zero_crossing_rate': mean_zcr,
            'spectral_rolloff': mean_rolloff,
            'dynamic_range_db': dynamic_range_db,
            'clipping_ratio': clipping_ratio
        }
    
    def normalize_audio(self, audio: np.ndarray) -> np.ndarray:
        """Intelligent audio normalization preserving dynamic range."""
        if not self.config.normalize:
            return audio
        
        # Peak normalization with headroom
        peak = np.max(np.abs(audio))
        if peak > 0:
            # Normalize to 90% of full scale to prevent clipping
            normalized = audio * (0.9 / peak)
        else:
            normalized = audio
        
        return normalized
    
    def is_valid_audio(self, metrics: Dict[str, float]) -> Tuple[bool, List[str]]:
        """Validate audio quality against configured thresholds."""
        issues = []
        
        # Duration checks
        if metrics['duration'] < self.config.min_duration:
            issues.append(f"Too short: {metrics['duration']:.2f}s < {self.config.min_duration}s")
        
        if metrics['duration'] > self.config.max_duration:
            issues.append(f"Too long: {metrics['duration']:.2f}s > {self.config.max_duration}s")
        
        # Quality checks
        if metrics['snr_db'] < self.config.min_snr_db:
            issues.append(f"Low SNR: {metrics['snr_db']:.1f}dB < {self.config.min_snr_db}dB")
        
        if metrics['zero_crossing_rate'] > self.config.max_zero_crossing_rate:
            issues.append(f"High ZCR: {metrics['zero_crossing_rate']:.3f} > {self.config.max_zero_crossing_rate}")
        
        if (metrics['spectral_centroid'] < self.config.min_spectral_centroid or 
            metrics['spectral_centroid'] > self.config.max_spectral_centroid):
            issues.append(f"Spectral centroid out of range: {metrics['spectral_centroid']:.0f}Hz")
        
        # Clipping detection
        if metrics['clipping_ratio'] > 0.01:  # More than 1% clipped samples
            issues.append(f"Audio clipping detected: {metrics['clipping_ratio']:.3f}")
        
        return len(issues) == 0, issues
    
    def process_audio_file(self, file_path: Union[str, Path]) -> Optional[Dict[str, Any]]:
        """Complete audio processing pipeline for a single file."""
        try:
            # Load audio
            audio, orig_sr = self.load_audio(file_path)
            if audio is None:
                return None
            
            # Resample
            audio = self.resample_audio(audio, orig_sr)
            
            # Apply VAD
            audio, vad_stats = self.apply_vad(audio)
            
            # Calculate quality metrics
            quality_metrics = self.calculate_quality_metrics(audio)
            
            # Validate audio
            is_valid, issues = self.is_valid_audio(quality_metrics)
            
            # Normalize if valid
            if is_valid:
                audio = self.normalize_audio(audio)
            
            # Combine all metrics
            result = {
                'file_path': str(file_path),
                'audio': audio if is_valid else None,
                'sample_rate': self.config.target_sr,
                'is_valid': is_valid,
                'issues': issues,
                **quality_metrics,
                **vad_stats
            }
            
            # Update statistics
            for key, value in quality_metrics.items():
                self.stats[key].append(value)
            
            return result
            
        except Exception as e:
            logger.error(f"Error processing {file_path}: {e}")
            return None
    
    def get_processing_stats(self) -> Dict[str, Any]:
        """Get comprehensive processing statistics."""
        stats = {}
        
        for metric, values in self.stats.items():
            if len(values) > 0:
                stats[metric] = {
                    'mean': np.mean(values),
                    'std': np.std(values),
                    'min': np.min(values),
                    'max': np.max(values),
                    'median': np.median(values),
                    'q25': np.percentile(values, 25),
                    'q75': np.percentile(values, 75)
                }
        
        return stats

print('✓ AudioProcessor class defined')
print('✓ Supports WebRTC VAD, spectral analysis, and quality assessment')

✓ AudioProcessor class defined
✓ Supports WebRTC VAD, spectral analysis, and quality assessment


---
## 4. Text Normalization Module <a id='section4'></a>

### Linguistic Theory for Kalenjin

Kalenjin text normalization requires understanding of:
- **Orthographic Variations**: Multiple spelling conventions exist
- **Tone Marking**: Optional tone diacritics in some texts
- **Morphophonemic Processes**: Sound changes at morpheme boundaries
- **Code-switching**: Mixed Kalenjin-English utterances

### Normalization Pipeline
1. **Unicode Normalization**: NFKC form for consistent representation
2. **Case Normalization**: Lowercase conversion with exceptions
3. **Punctuation Handling**: Remove while preserving apostrophes
4. **Orthographic Standardization**: Consistent spelling rules
5. **Quality Filtering**: Remove low-quality transcriptions

In [4]:
class TextNormalizer:
    """Advanced text normalization for Kalenjin ASR preprocessing."""
    
    def __init__(self, config: TextConfig):
        self.config = config
        self.stats = defaultdict(int)
        
        # Kalenjin-specific orthographic mappings
        self.orthographic_rules = {
            'ch': 'c',  # Standardize consonant clusters
            # I want to preserve the important apostrophe in "ng'", which is a common feature in Kalenjin orthography
            "ng'": "ng'",  # Preserve important apostrophe
            'kh': 'k',  # Simplify aspirated consonants
        }
        
        # Common abbreviations and expansions
        self.expansions = {
            'n': 'na',  # Common conjunction
            'k': 'ko',  # Common preposition
        }
        
        logger.info("TextNormalizer initialized for Kalenjin")
    
    def normalize_unicode(self, text: str) -> str:
        """Normalize Unicode representation."""
        # NFKC normalization for consistent representation
        normalized = unicodedata.normalize('NFKC', text)
        
        # Remove zero-width characters
        normalized = re.sub(r'[​-‏﻿]', '', normalized)
        
        return normalized
    
    def normalize_case(self, text: str) -> str:
        """Apply case normalization with language-specific rules."""
        if self.config.lowercase:
            return text.lower()
        return text
    
    def clean_punctuation(self, text: str) -> str:
        """Remove punctuation while preserving important markers."""
        if not self.config.remove_punctuation:
            return text
        
        # Preserve apostrophes if configured
        if self.config.preserve_apostrophes:
            # Replace other punctuation but keep apostrophes
            text = re.sub(r"[^\w\s']", ' ', text)
        else:
            # Remove all punctuation
            text = re.sub(r'[^\w\s]', ' ', text)
        
        return text
    
    def apply_orthographic_rules(self, text: str) -> str:
        """Apply Kalenjin-specific orthographic standardization."""
        if not self.config.standardize_orthography:
            return text
        
        # Apply orthographic mappings
        for old, new in self.orthographic_rules.items():
            text = text.replace(old, new)
        
        return text
    
    def expand_abbreviations(self, text: str) -> str:
        """Expand common abbreviations."""
        words = text.split()
        expanded_words = []
        
        for word in words:
            if word in self.expansions:
                expanded_words.append(self.expansions[word])
                self.stats['abbreviations_expanded'] += 1
            else:
                expanded_words.append(word)
        
        return ' '.join(expanded_words)
    
    def filter_characters(self, text: str) -> str:
        """Filter to allowed character set."""
        # Create allowed character set
        allowed = set(self.config.allowed_chars)
        if self.config.preserve_apostrophes:
            allowed.add("'")
        
        # Filter characters
        filtered_chars = [c for c in text if c in allowed]
        filtered_text = ''.join(filtered_chars)
        
        # Normalize whitespace
        filtered_text = re.sub(r'\s+', ' ', filtered_text).strip()
        
        return filtered_text
    
    def detect_repetitions(self, text: str) -> int:
        """Detect excessive character repetitions."""
        max_repetition = 0
        current_char = ''
        current_count = 0
        
        for char in text:
            if char == current_char:
                current_count += 1
                max_repetition = max(max_repetition, current_count)
            else:
                current_char = char
                current_count = 1
        
        return max_repetition
    
    def calculate_text_metrics(self, text: str) -> Dict[str, Any]:
        """Calculate comprehensive text quality metrics."""
        words = text.split()
        
        metrics = {
            'char_count': len(text),
            'word_count': len(words),
            'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
            'max_repetition': self.detect_repetitions(text),
            'unique_chars': len(set(text.lower())),
            'whitespace_ratio': text.count(' ') / len(text) if text else 0,
        }
        
        return metrics
    
    def is_valid_text(self, text: str, metrics: Dict[str, Any]) -> Tuple[bool, List[str]]:
        """Validate text quality against configured thresholds."""
        issues = []
        
        # Word count checks
        if metrics['word_count'] < self.config.min_words:
            issues.append(f"Too few words: {metrics['word_count']} < {self.config.min_words}")
        
        if metrics['word_count'] > self.config.max_words:
            issues.append(f"Too many words: {metrics['word_count']} > {self.config.max_words}")
        
        # Character repetition check
        if metrics['max_repetition'] > self.config.max_char_repetition:
            issues.append(f"Excessive repetition: {metrics['max_repetition']} > {self.config.max_char_repetition}")
        
        # Empty text check
        if not text.strip():
            issues.append("Empty text after normalization")
        
        return len(issues) == 0, issues
    
    def normalize_text(self, text: str) -> Dict[str, Any]:
        """Complete text normalization pipeline."""
        if not isinstance(text, str):
            return {
                'original': text,
                'normalized': '',
                'is_valid': False,
                'issues': ['Input is not a string'],
                'metrics': {}
            }
        
        original_text = text
        
        try:
            # Normalization pipeline
            text = self.normalize_unicode(text)
            text = self.normalize_case(text)
            text = self.clean_punctuation(text)
            text = self.apply_orthographic_rules(text)
            text = self.expand_abbreviations(text)
            text = self.filter_characters(text)
            
            # Calculate metrics
            metrics = self.calculate_text_metrics(text)
            
            # Validate
            is_valid, issues = self.is_valid_text(text, metrics)
            
            # Update statistics
            self.stats['total_processed'] += 1
            if is_valid:
                self.stats['valid_texts'] += 1
            else:
                self.stats['invalid_texts'] += 1
            
            return {
                'original': original_text,
                'normalized': text,
                'is_valid': is_valid,
                'issues': issues,
                'metrics': metrics
            }
            
        except Exception as e:
            logger.error(f"Error normalizing text '{text[:50]}...': {e}")
            return {
                'original': original_text,
                'normalized': '',
                'is_valid': False,
                'issues': [f'Processing error: {str(e)}'],
                'metrics': {}
            }
    
    def get_processing_stats(self) -> Dict[str, Any]:
        """Get text processing statistics."""
        total = self.stats['total_processed']
        if total == 0:
            return {'message': 'No texts processed yet'}
        
        return {
            'total_processed': total,
            'valid_texts': self.stats['valid_texts'],
            'invalid_texts': self.stats['invalid_texts'],
            'validity_rate': self.stats['valid_texts'] / total,
            'abbreviations_expanded': self.stats['abbreviations_expanded']
        }

print('✓ TextNormalizer class defined')
print('✓ Supports Kalenjin-specific orthographic rules')
print('✓ Includes comprehensive text quality validation')

✓ TextNormalizer class defined
✓ Supports Kalenjin-specific orthographic rules
✓ Includes comprehensive text quality validation


---
## 5. Quality Assessment Module <a id='section5'></a>

### Comprehensive Quality Framework

The quality assessment module implements multi-dimensional evaluation:
- **Audio Quality**: SNR, spectral analysis, clipping detection
- **Text Quality**: Linguistic validity, character analysis
- **Alignment Quality**: Audio-text correspondence metrics
- **Statistical Analysis**: Outlier detection and quality distributions

In [5]:
# Import quality assessment module
import sys
sys.path.append('../scripts')
from quality_assessment import AudioQualityAssessor, TextQualityAssessor, QualityThresholds

# Initialize quality assessors
quality_thresholds = QualityThresholds()
audio_assessor = AudioQualityAssessor(quality_thresholds)
text_assessor = TextQualityAssessor(quality_thresholds)

print('✓ Quality assessment module loaded')
print('✓ Audio and text quality assessors initialized')

✓ Quality assessment module loaded
✓ Audio and text quality assessors initialized


---
## 6. Dataset Structuring Module <a id='section6'></a>

### Dataset Organization Strategy

Optimal dataset structuring for ASR training requires:
- **Stratified Sampling**: Balanced representation across speakers/domains
- **Duration Distribution**: Optimal length distribution for training
- **Train/Validation/Test Splits**: Proper data leakage prevention
- **Metadata Management**: Comprehensive sample annotations

In [6]:
class DatasetStructurer:
    """Advanced dataset structuring for ASR training."""
    
    def __init__(self, config: ProcessingConfig):
        self.config = config
        self.metadata = {}
        
    def create_splits(self, data: List[Dict], test_size: float = 0.2, val_size: float = 0.1) -> Dict[str, List]:
        """Create stratified train/validation/test splits."""
        # First split: train+val vs test
        train_val, test = train_test_split(
            data, 
            test_size=test_size, 
            random_state=self.config.random_seed
        )
        
        # Second split: train vs val
        train, val = train_test_split(
            train_val, 
            test_size=val_size/(1-test_size), 
            random_state=self.config.random_seed
        )
        
        return {
            'train': train,
            'validation': val,
            'test': test
        }
    
    def create_huggingface_dataset(self, splits: Dict[str, List]) -> DatasetDict:
        """Create HuggingFace Dataset format."""
        dataset_dict = {}
        
        for split_name, split_data in splits.items():
            # Prepare data for HuggingFace format
            hf_data = {
                'audio': [item['audio'] for item in split_data],
                'text': [item['normalized_text'] for item in split_data],
                'duration': [item['duration'] for item in split_data]
            }
            
            dataset_dict[split_name] = Dataset.from_dict(hf_data)
            dataset_dict[split_name] = dataset_dict[split_name].cast_column('audio', Audio(sampling_rate=16000))
        
        return DatasetDict(dataset_dict)

print('✓ DatasetStructurer class defined')

✓ DatasetStructurer class defined


---
## 7. Pipeline Orchestration <a id='section7'></a>

### Modular Pipeline Architecture

The orchestration system coordinates all preprocessing modules:
- **Parallel Processing**: Multi-core audio/text processing
- **Memory Management**: Efficient batch processing
- **Error Handling**: Robust failure recovery
- **Progress Tracking**: Comprehensive logging and monitoring

In [7]:
class PreprocessingPipeline:
    """Master preprocessing pipeline orchestrator."""
    
    def __init__(self, config: PreprocessingConfig):
        self.config = config
        self.audio_processor = AudioProcessor(config.audio)
        self.text_normalizer = TextNormalizer(config.text)
        self.dataset_structurer = DatasetStructurer(config.processing)
        self.results = []
        
    def process_sample(self, audio_path: str, text: str) -> Optional[Dict]:
        """Process a single audio-text pair."""
        try:
            # Process audio
            audio_result = self.audio_processor.process_audio_file(audio_path)
            if not audio_result or not audio_result['is_valid']:
                return None
            
            # Process text
            text_result = self.text_normalizer.normalize_text(text)
            if not text_result['is_valid']:
                return None
            
            # Combine results
            return {
                'audio_path': audio_path,
                'audio': audio_result['audio'],
                'sample_rate': audio_result['sample_rate'],
                'duration': audio_result['duration'],
                'original_text': text,
                'normalized_text': text_result['normalized'],
                'audio_metrics': {k: v for k, v in audio_result.items() if k not in ['audio', 'file_path']},
                'text_metrics': text_result['metrics']
            }
            
        except Exception as e:
            logger.error(f"Error processing sample {audio_path}: {e}")
            return None
    

    
    def run_pipeline(self, data_pairs: List[Tuple[str, str]]) -> Dict[str, Any]:
        """Run complete preprocessing pipeline."""
        logger.info(f"Starting pipeline with {len(data_pairs)} samples")
        
        # Process samples sequentially
        results = []
        for audio_path, text in tqdm(data_pairs, desc="Processing samples"):
            result = self.process_sample(audio_path, text)
            results.append(result)
        
        # Filter valid results
        valid_results = [r for r in results if r is not None]
        
        logger.info(f"Pipeline completed: {len(valid_results)}/{len(data_pairs)} samples valid")
        
        return {
            'processed_data': valid_results,
            'total_samples': len(data_pairs),
            'valid_samples': len(valid_results),
            'success_rate': len(valid_results) / len(data_pairs) if data_pairs else 0
        }


print('✓ PreprocessingPipeline class defined')

✓ PreprocessingPipeline class defined


---
## 8. Validation & Quality Control <a id='section8'></a>

### Multi-Level Validation Framework

Comprehensive validation ensures data quality:
- **Statistical Validation**: Distribution analysis and outlier detection
- **Linguistic Validation**: Text quality and consistency checks
- **Audio Validation**: Signal quality and technical specifications
- **Cross-Validation**: Audio-text alignment verification

In [8]:
class QualityValidator:
    """Comprehensive quality validation system."""
    
    def __init__(self, config: PreprocessingConfig):
        self.config = config
        
    def validate_dataset_statistics(self, data: List[Dict]) -> Dict[str, Any]:
        """Validate dataset statistical properties."""
        durations = [item['duration'] for item in data]
        word_counts = [item['text_metrics']['word_count'] for item in data]
        
        stats = {
            'duration_stats': {
                'mean': np.mean(durations),
                'std': np.std(durations),
                'min': np.min(durations),
                'max': np.max(durations),
                'total_hours': np.sum(durations) / 3600
            },
            'text_stats': {
                'mean_words': np.mean(word_counts),
                'std_words': np.std(word_counts),
                'total_words': np.sum(word_counts)
            }
        }
        
        return stats
    
    def detect_outliers(self, data: List[Dict]) -> List[int]:
        """Detect statistical outliers in the dataset."""
        durations = np.array([item['duration'] for item in data])
        
        # Use IQR method for outlier detection
        Q1 = np.percentile(durations, 25)
        Q3 = np.percentile(durations, 75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outlier_indices = np.where((durations < lower_bound) | (durations > upper_bound))[0]
        
        return outlier_indices.tolist()

print('✓ QualityValidator class defined')

✓ QualityValidator class defined


---
## 9. Export & Model Preparation <a id='section9'></a>

### Multi-Format Export System

The export module supports multiple ASR frameworks:
- **HuggingFace Datasets**: Direct integration with Transformers
- **JSON Format**: Flexible metadata preservation
- **CSV Format**: Tabular data analysis
- **Model-Specific**: Optimized formats for different architectures

In [9]:
class DataExporter:
    """Multi-format data export system."""
    
    def __init__(self, config: ProcessingConfig):
        self.config = config
        
    def export_huggingface(self, dataset: DatasetDict, output_dir: Path) -> None:
        """Export to HuggingFace Dataset format."""
        output_dir.mkdir(parents=True, exist_ok=True)
        dataset.save_to_disk(str(output_dir / 'huggingface_dataset'))
        logger.info(f"HuggingFace dataset saved to {output_dir / 'huggingface_dataset'}")
    
    def export_json(self, data: List[Dict], output_path: Path) -> None:
        """Export to JSON format."""
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        logger.info(f"JSON data saved to {output_path}")
    
    def export_csv(self, data: List[Dict], output_path: Path) -> None:
        """Export to CSV format."""
        # Flatten nested dictionaries for CSV
        flattened_data = []
        for item in data:
            flat_item = {
                'audio_path': item['audio_path'],
                'duration': item['duration'],
                'original_text': item['original_text'],
                'normalized_text': item['normalized_text']
            }
            # Add metrics
            for key, value in item['audio_metrics'].items():
                flat_item[f'audio_{key}'] = value
            for key, value in item['text_metrics'].items():
                flat_item[f'text_{key}'] = value
            flattened_data.append(flat_item)
        
        df = pd.DataFrame(flattened_data)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(output_path, index=False)
        logger.info(f"CSV data saved to {output_path}")

print('✓ DataExporter class defined')

✓ DataExporter class defined


---
## 10. Performance Analysis <a id='section10'></a>

### Comprehensive Performance Metrics

The performance analysis module provides:
- **Processing Statistics**: Throughput and efficiency metrics
- **Quality Distributions**: Statistical analysis of data quality
- **Visualization**: Interactive plots and reports
- **Benchmarking**: Performance comparison across configurations

In [10]:
class PerformanceAnalyzer:
    """Comprehensive performance analysis and visualization."""
    
    def __init__(self):
        self.metrics = {}
        
    def analyze_processing_performance(self, results: Dict[str, Any]) -> Dict[str, Any]:
        """Analyze preprocessing performance metrics."""
        data = results['processed_data']
        
        # Duration analysis
        durations = [item['duration'] for item in data]
        total_audio_hours = sum(durations) / 3600
        
        # Quality metrics
        snr_values = [item['audio_metrics']['snr_db'] for item in data]
        word_counts = [item['text_metrics']['word_count'] for item in data]
        
        analysis = {
            'dataset_summary': {
                'total_samples': len(data),
                'total_audio_hours': total_audio_hours,
                'success_rate': results['success_rate'],
                'avg_duration': np.mean(durations),
                'avg_snr': np.mean(snr_values),
                'avg_words_per_sample': np.mean(word_counts)
            },
            'quality_distribution': {
                'duration_percentiles': {
                    'p10': np.percentile(durations, 10),
                    'p25': np.percentile(durations, 25),
                    'p50': np.percentile(durations, 50),
                    'p75': np.percentile(durations, 75),
                    'p90': np.percentile(durations, 90)
                },
                'snr_percentiles': {
                    'p10': np.percentile(snr_values, 10),
                    'p25': np.percentile(snr_values, 25),
                    'p50': np.percentile(snr_values, 50),
                    'p75': np.percentile(snr_values, 75),
                    'p90': np.percentile(snr_values, 90)
                }
            }
        }
        
        return analysis
    
    def create_quality_visualizations(self, data: List[Dict]) -> None:
        """Create comprehensive quality visualization plots."""
        # Extract metrics
        durations = [item['duration'] for item in data]
        snr_values = [item['audio_metrics']['snr_db'] for item in data]
        word_counts = [item['text_metrics']['word_count'] for item in data]
        
        # Create subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Kalenjin ASR Dataset Quality Analysis', fontsize=16)
        
        # Duration distribution
        axes[0, 0].hist(durations, bins=50, alpha=0.7, color='skyblue')
        axes[0, 0].set_title('Audio Duration Distribution')
        axes[0, 0].set_xlabel('Duration (seconds)')
        axes[0, 0].set_ylabel('Frequency')
        
        # SNR distribution
        axes[0, 1].hist(snr_values, bins=50, alpha=0.7, color='lightgreen')
        axes[0, 1].set_title('Signal-to-Noise Ratio Distribution')
        axes[0, 1].set_xlabel('SNR (dB)')
        axes[0, 1].set_ylabel('Frequency')
        
        # Word count distribution
        axes[1, 0].hist(word_counts, bins=30, alpha=0.7, color='salmon')
        axes[1, 0].set_title('Word Count Distribution')
        axes[1, 0].set_xlabel('Words per Sample')
        axes[1, 0].set_ylabel('Frequency')
        
        # Duration vs SNR scatter
        axes[1, 1].scatter(durations, snr_values, alpha=0.6, color='purple')
        axes[1, 1].set_title('Duration vs SNR Relationship')
        axes[1, 1].set_xlabel('Duration (seconds)')
        axes[1, 1].set_ylabel('SNR (dB)')
        
        plt.tight_layout()
        plt.show()
    
    def generate_report(self, analysis: Dict[str, Any], output_path: Path) -> None:
        """Generate comprehensive analysis report."""
        report = f"""
KALENJIN ASR PREPROCESSING PERFORMANCE REPORT
{'='*50}

DATASET SUMMARY
{'-'*20}
Total Samples: {analysis['dataset_summary']['total_samples']:,}
Total Audio Hours: {analysis['dataset_summary']['total_audio_hours']:.2f}
Success Rate: {analysis['dataset_summary']['success_rate']:.1%}
Average Duration: {analysis['dataset_summary']['avg_duration']:.2f}s
Average SNR: {analysis['dataset_summary']['avg_snr']:.1f}dB
Average Words/Sample: {analysis['dataset_summary']['avg_words_per_sample']:.1f}

QUALITY DISTRIBUTION
{'-'*20}
Duration Percentiles (seconds):
  P10: {analysis['quality_distribution']['duration_percentiles']['p10']:.2f}
  P25: {analysis['quality_distribution']['duration_percentiles']['p25']:.2f}
  P50: {analysis['quality_distribution']['duration_percentiles']['p50']:.2f}
  P75: {analysis['quality_distribution']['duration_percentiles']['p75']:.2f}
  P90: {analysis['quality_distribution']['duration_percentiles']['p90']:.2f}

SNR Percentiles (dB):
  P10: {analysis['quality_distribution']['snr_percentiles']['p10']:.1f}
  P25: {analysis['quality_distribution']['snr_percentiles']['p25']:.1f}
  P50: {analysis['quality_distribution']['snr_percentiles']['p50']:.1f}
  P75: {analysis['quality_distribution']['snr_percentiles']['p75']:.1f}
  P90: {analysis['quality_distribution']['snr_percentiles']['p90']:.1f}
"""
        
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w') as f:
            f.write(report)
        
        logger.info(f"Performance report saved to {output_path}")

print('✓ PerformanceAnalyzer class defined')
print('✓ All preprocessing modules loaded successfully')
print('✓ Kalenjin ASR preprocessing pipeline ready for execution')

✓ PerformanceAnalyzer class defined
✓ All preprocessing modules loaded successfully
✓ Kalenjin ASR preprocessing pipeline ready for execution


In [11]:
# ============================================================================
# STEP 2: Process Test Batch (100 samples)
# ============================================================================

print("="*70)
print("PROCESSING TEST BATCH".center(70))
print("="*70)

# Load small batch
DATA_PATH = Path('../cv-corpus-24.0-2025-12-05-kln/cv-corpus-24.0-2025-12-05/kln')
train_df = pd.read_csv(DATA_PATH / 'train.tsv', sep='\t', nrows=100)

# Prepare audio-text pairs
data_pairs = []
for idx, row in train_df.iterrows():
    audio_path = DATA_PATH / 'clips' / row['path']
    if audio_path.exists():
        data_pairs.append((str(audio_path), row['sentence']))

print(f"\nFound {len(data_pairs)} valid audio files")

# Initialize pipeline
config = PreprocessingConfig(
    audio=AudioConfig(),
    text=TextConfig(),
    processing=ProcessingConfig()
)
pipeline = PreprocessingPipeline(config)

# Process batch
print("\nProcessing...")
results = pipeline.run_pipeline(data_pairs)

# Display results
print("\n" + "="*70)
print("RESULTS".center(70))
print("="*70)
print(f"Total samples:   {results['total_samples']}")
print(f"Valid samples:   {results['valid_samples']}")
print(f"Success rate:    {results['success_rate']:.1%}")
print("="*70)

# Quick quality check
if results['valid_samples'] > 0:
    sample_data = results['processed_data'][0]
    print("\nSample output:")
    print(f"  Duration: {sample_data['duration']:.2f}s")
    print(f"  SNR: {sample_data['audio_metrics']['snr_db']:.1f}dB")
    print(f"  Text: {sample_data['normalized_text']}")


                        PROCESSING TEST BATCH                         

Found 100 valid audio files


2026-02-10 21:34:36,101 - INFO - AudioProcessor initialized with SR=16000Hz
2026-02-10 21:34:36,102 - INFO - TextNormalizer initialized for Kalenjin
2026-02-10 21:34:36,102 - INFO - Starting pipeline with 100 samples



Processing...


Processing samples: 100%|██████████| 100/100 [00:05<00:00, 16.84it/s]
2026-02-10 21:34:42,045 - INFO - Pipeline completed: 100/100 samples valid



                               RESULTS                                
Total samples:   100
Valid samples:   100
Success rate:    100.0%

Sample output:
  Duration: 2.14s
  SNR: 45.9dB
  Text: tomo itinye coruet ne cepto iman


In [ ]:
def convert_to_json_serializable(obj):
    """Convert numpy types to native Python types."""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_json_serializable(item) for item in obj]
    return obj

print('✓ JSON serialization helper loaded')


In [15]:
# Create output directory
Path('../processed_data').mkdir(parents=True, exist_ok=True)

# Process all splits
for split in ['train', 'dev', 'test']:
    print(f"\nProcessing {split} split...")
    
    df = pd.read_csv(DATA_PATH / f'{split}.tsv', sep='\t')
    pairs = [(str(DATA_PATH / 'clips' / row['path']), row['sentence']) 
             for _, row in df.iterrows() 
             if (DATA_PATH / 'clips' / row['path']).exists()]
    
    results = pipeline.run_pipeline(pairs)
    
    # Remove audio arrays and convert numpy types
    results_to_save = {
        'total_samples': results['total_samples'],
        'valid_samples': results['valid_samples'],
        'success_rate': results['success_rate'],
        'metadata': [
            {k: v for k, v in item.items() if k != 'audio'}
            for item in results['processed_data']
        ]
    }
    
    # Convert numpy types to native Python types
    results_to_save = convert_to_json_serializable(results_to_save)
    
    # Save results
    with open(f'../processed_data/{split}_results.json', 'w') as f:
        json.dump(results_to_save, f, indent=2)
    
    print(f"{split}: {results['success_rate']:.1%} success ({results['valid_samples']}/{results['total_samples']})")

print("\n✓ All splits processed and saved successfully!")



Processing train split...


2026-02-10 21:57:22,911 - INFO - Starting pipeline with 11065 samples
Processing samples: 100%|██████████| 11065/11065 [05:03<00:00, 36.49it/s]
2026-02-10 22:02:26,169 - INFO - Pipeline completed: 11057/11065 samples valid


train: 99.9% success (11057/11065)

Processing dev split...


2026-02-10 22:02:27,419 - INFO - Starting pipeline with 6412 samples
Processing samples: 100%|██████████| 6412/6412 [03:10<00:00, 33.63it/s]
2026-02-10 22:05:38,091 - INFO - Pipeline completed: 6409/6412 samples valid


dev: 100.0% success (6409/6412)

Processing test split...


2026-02-10 22:05:38,959 - INFO - Starting pipeline with 5685 samples
Processing samples: 100%|██████████| 5685/5685 [03:20<00:00, 28.36it/s]
2026-02-10 22:08:59,434 - INFO - Pipeline completed: 5675/5685 samples valid


test: 99.8% success (5675/5685)

✓ All splits processed and saved successfully!


In [16]:
# Generate Summary Statistics
# Load and analyze results
import json
from pathlib import Path

splits_summary = {}
for split in ['train', 'dev', 'test']:
    with open(f'../processed_data/{split}_results.json', 'r') as f:
        data = json.load(f)
        splits_summary[split] = {
            'samples': data['valid_samples'],
            'success_rate': data['success_rate']
        }

print("Dataset Summary:")
print("=" * 50)
for split, stats in splits_summary.items():
    print(f"{split.upper():8s}: {stats['samples']:5d} samples ({stats['success_rate']:.1%})")
print("=" * 50)
print(f"TOTAL:   {sum(s['samples'] for s in splits_summary.values())} samples")


Dataset Summary:
TRAIN   : 11057 samples (99.9%)
DEV     :  6409 samples (100.0%)
TEST    :  5675 samples (99.8%)
TOTAL:   23141 samples


In [17]:
# Create a HuggingFace Dataset Format
# Create HuggingFace dataset with audio files
from datasets import Dataset, DatasetDict, Audio, Features, Value

def create_hf_dataset(split):
    with open(f'../processed_data/{split}_results.json', 'r') as f:
        data = json.load(f)
    
    dataset_dict = {
        'audio': [item['audio_path'] for item in data['metadata']],
        'text': [item['normalized_text'] for item in data['metadata']],
        'duration': [item['duration'] for item in data['metadata']],
    }
    
    dataset = Dataset.from_dict(dataset_dict)
    dataset = dataset.cast_column('audio', Audio(sampling_rate=16000))
    return dataset

# Create dataset dict
dataset_dict = DatasetDict({
    'train': create_hf_dataset('train'),
    'validation': create_hf_dataset('dev'),
    'test': create_hf_dataset('test')
})

# Save to disk
dataset_dict.save_to_disk('../processed_data/kalenjin_asr_dataset')
print("✓ HuggingFace dataset saved!")


Saving the dataset (1/1 shards): 100%|██████████| 5675/5675 [00:00<00:00, 5768.52 examples/s]

✓ HuggingFace dataset saved!


In [18]:
# Generate Quality Report
# Analyze quality metrics
for split in ['train', 'dev', 'test']:
    with open(f'../processed_data/{split}_results.json', 'r') as f:
        data = json.load(f)
    
    durations = [item['duration'] for item in data['metadata']]
    snr_values = [item['audio_metrics']['snr_db'] for item in data['metadata']]
    
    print(f"\n{split.upper()} Quality Metrics:")
    print(f"  Duration: {np.mean(durations):.2f}s ± {np.std(durations):.2f}s")
    print(f"  SNR: {np.mean(snr_values):.1f}dB ± {np.std(snr_values):.1f}dB")
    print(f"  Total hours: {sum(durations)/3600:.2f}h")



TRAIN Quality Metrics:
  Duration: 2.84s ± 1.30s
  SNR: 50.7dB ± 14.9dB
  Total hours: 8.71h

DEV Quality Metrics:
  Duration: 3.02s ± 1.42s
  SNR: 48.9dB ± 13.3dB
  Total hours: 5.38h

TEST Quality Metrics:
  Duration: 3.89s ± 2.17s
  SNR: 55.5dB ± 12.1dB
  Total hours: 6.13h


In [20]:
# Create HuggingFace Dataset
from datasets import Dataset, DatasetDict, Audio

def create_hf_dataset(split):
    with open(f'../processed_data/{split}_results.json', 'r') as f:
        data = json.load(f)
    
    return Dataset.from_dict({
        'audio': [item['audio_path'] for item in data['metadata']],
        'text': [item['normalized_text'] for item in data['metadata']],
        'duration': [item['duration'] for item in data['metadata']],
    }).cast_column('audio', Audio(sampling_rate=16000))

dataset_dict = DatasetDict({
    'train': create_hf_dataset('train'),
    'validation': create_hf_dataset('dev'),
    'test': create_hf_dataset('test')
})

dataset_dict.save_to_disk('../processed_data/kalenjin_asr_dataset')
print(f"✓ Dataset saved: {len(dataset_dict['train'])} train, {len(dataset_dict['validation'])} val, {len(dataset_dict['test'])} test")


Saving the dataset (1/1 shards): 100%|██████████| 5675/5675 [00:02<00:00, 2820.44 examples/s]

✓ Dataset saved: 11057 train, 6409 val, 5675 test


In [21]:
print("="*70)
print("CREATING HUGGINGFACE DATASET".center(70))
print("="*70)

from datasets import Dataset, DatasetDict, Audio

def create_hf_dataset(split):
    """Load JSON metadata and create HF dataset."""
    with open(f'../processed_data/{split}_results.json', 'r') as f:
        data = json.load(f)
    
    dataset_dict = {
        'audio': [item['audio_path'] for item in data['metadata']],
        'text': [item['normalized_text'] for item in data['metadata']],
        'duration': [item['duration'] for item in data['metadata']],
        'snr_db': [item['audio_metrics']['snr_db'] for item in data['metadata']],
    }
    
    dataset = Dataset.from_dict(dataset_dict)
    dataset = dataset.cast_column('audio', Audio(sampling_rate=16000))
    return dataset

# Create dataset for all splits
print("\nCreating datasets...")
dataset_dict = DatasetDict({
    'train': create_hf_dataset('train'),
    'validation': create_hf_dataset('dev'),
    'test': create_hf_dataset('test')
})

# Save to disk
output_path = Path('../processed_data/kalenjin_asr_dataset')
dataset_dict.save_to_disk(str(output_path))

print(f"\n✓ Dataset saved to: {output_path}")
print(f"  Train:      {len(dataset_dict['train']):,} samples")
print(f"  Validation: {len(dataset_dict['validation']):,} samples")
print(f"  Test:       {len(dataset_dict['test']):,} samples")
print(f"  Total:      {sum(len(dataset_dict[s]) for s in dataset_dict):,} samples")


                     CREATING HUGGINGFACE DATASET                     

Creating datasets...


Saving the dataset (1/1 shards): 100%|██████████| 5675/5675 [00:01<00:00, 3093.12 examples/s]


✓ Dataset saved to: ../processed_data/kalenjin_asr_dataset
  Train:      11,057 samples
  Validation: 6,409 samples
  Test:       5,675 samples
  Total:      23,141 samples


In [9]:
from datasets import load_from_disk
import re
import json
from pathlib import Path
from collections import Counter

# Load dataset
dataset_dict = load_from_disk('../processed_data/kalenjin_asr_dataset')

print("\n" + "="*70)
print("BUILDING VOCABULARY - RESEARCH PRECISION".center(70))
print("="*70)

def clean_kalenjin_text(text):
    """Normalize Kalenjin text for CTC training with orthographic preservation."""
    text = text.lower().strip()
    
    # CORRECTED: Target actual curly/smart apostrophes and diverse glottal markers
    text = text.replace("’", "'").replace("‘", "'").replace("`", "'").replace("´", "'")
    
    # Keep only standard letters, the normalized apostrophe, and spaces
    text = re.sub(r"[^a-z' ]", "", text)
    
    # Collapse multiple spaces to ensure clean CTC alignment
    text = re.sub(r'\s+', ' ', text)
    return text

# Fixed index mapping (Research Standard for CTC)
vocab_dict = {
    "[PAD]": 0,
    "[UNK]": 1,
    "[CTC]": 2,  # The blank token
    "|": 3,      # The word boundary
}

# Analyze corpus
all_texts = " ".join(dataset_dict['train']['text'])
cleaned_text = clean_kalenjin_text(all_texts)
char_freq = Counter(cleaned_text)

# Add characters alphabetically starting from index 4
chars = sorted([c for c in char_freq.keys() if c not in vocab_dict])
for char in chars:
    vocab_dict[char] = len(vocab_dict)

# Save vocabulary
vocab_path = Path('../processed_data/vocab.json')
vocab_path.parent.mkdir(parents=True, exist_ok=True) # SAFETY: Create dir if missing
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

print(f"\n✓ Vocabulary size: {len(vocab_dict)}")
print(f"✓ Characters: {' '.join(chars)}")

# Transform dataset
def prepare_dataset(batch):
    # Apply normalization and convert spaces to CTC pipes |
    batch['text'] = [clean_kalenjin_text(t).replace(' ', '|') for t in batch['text']]
    return batch

print(f"\n✓ Transforming dataset with 4 parallel processes...")
dataset_dict = dataset_dict.map(prepare_dataset, batched=True, num_proc=4)
print(f"✓ Complete")


               BUILDING VOCABULARY - RESEARCH PRECISION               

✓ Vocabulary size: 32
✓ Characters:   ' a b c d e f g h i j k l m n o p q r s t u v w x y z

✓ Transforming dataset with 4 parallel processes...


Map (num_proc=4): 100%|██████████| 5675/5675 [00:01<00:00, 3420.90 examples/s]

✓ Complete


In [23]:
print("\n" + "="*70)
print("DATASET STATISTICS".center(70))
print("="*70)

for split_name in ['train', 'validation', 'test']:
    split = dataset_dict[split_name]
    durations = split['duration']
    
    total_hours = sum(durations) / 3600
    avg_duration = np.mean(durations)
    std_duration = np.std(durations)
    
    print(f"\n{split_name.upper()}:")
    print(f"  Samples:       {len(split):,}")
    print(f"  Total hours:   {total_hours:.2f}h")
    print(f"  Avg duration:  {avg_duration:.2f}s ± {std_duration:.2f}s")
    print(f"  Min duration:  {min(durations):.2f}s")
    print(f"  Max duration:  {max(durations):.2f}s")

# Overall statistics
total_samples = sum(len(dataset_dict[s]) for s in dataset_dict)
total_hours = sum(sum(dataset_dict[s]['duration']) for s in dataset_dict) / 3600

print(f"\n{'='*70}")
print(f"TOTAL: {total_samples:,} samples | {total_hours:.2f} hours")
print(f"{'='*70}")



                          DATASET STATISTICS                          

TRAIN:
  Samples:       11,057
  Total hours:   8.71h
  Avg duration:  2.84s ± 1.30s
  Min duration:  0.51s
  Max duration:  12.03s

VALIDATION:
  Samples:       6,409
  Total hours:   5.38h
  Avg duration:  3.02s ± 1.42s
  Min duration:  0.58s
  Max duration:  11.71s

TEST:
  Samples:       5,675
  Total hours:   6.13h
  Avg duration:  3.89s ± 2.17s
  Min duration:  0.51s
  Max duration:  15.02s

TOTAL: 23,141 samples | 20.22 hours


In [25]:
print("\n" + "="*70)
print("DATASET VERIFICATION".center(70))
print("="*70)

# Load dataset from disk to verify
from datasets import load_from_disk

# Load without automatic audio decoding to avoid torchcodec dependency
loaded_dataset = load_from_disk('../processed_data/kalenjin_asr_dataset')

print("\n✓ Dataset loaded successfully!")
print(f"\nSample from training set:")

# Set decode=False to avoid the torchcodec requirement
loaded_dataset = loaded_dataset.cast_column("audio", Audio(decode=False))
sample = loaded_dataset['train'][0]

print(f"  Text: {sample['text']}")
print(f"  Duration: {sample['duration']:.2f}s")
print(f"  SNR: {sample['snr_db']:.1f}dB")

# For audio, just show the path since we're not decoding
if 'path' in sample['audio']:
    print(f"  Audio path: {sample['audio']['path']}")
else:
    print(f"  Audio bytes: {len(sample['audio']['bytes'])} bytes")

print("\n✓ All preprocessing complete!")



                         DATASET VERIFICATION                         

✓ Dataset loaded successfully!

Sample from training set:
  Text: tomo itinye coruet ne cepto iman
  Duration: 2.14s
  SNR: 45.9dB
  Audio path: common_voice_kln_40550981.mp3

✓ All preprocessing complete!


In [5]:
print("\n" + "="*70)
print("DATASET VERIFICATION".center(70))
print("="*70)

from datasets import load_from_disk

# Load dataset
loaded_dataset = load_from_disk('../processed_data/kalenjin_asr_dataset')

print("\n✓ Dataset loaded successfully!")

# Access raw data without decoding
train_table = loaded_dataset['train']._data.table
sample_idx = 0

print(f"\nSample from training set:")
print(f"  Text: {train_table['text'][sample_idx].as_py()}")
print(f"  Duration: {train_table['duration'][sample_idx].as_py():.2f}s")
print(f"  SNR: {train_table['snr_db'][sample_idx].as_py():.1f}dB")

# Check audio column structure
audio_col = train_table['audio'][sample_idx]
audio_bytes = audio_col['bytes'].as_py() if audio_col['bytes'].as_py() else None
if audio_bytes:
    print(f"  Audio data: {len(audio_bytes)} bytes stored")
else:
    print(f"  Audio path: {audio_col['path'].as_py()}")

print("\n✓ All preprocessing complete!")



                         DATASET VERIFICATION                         

✓ Dataset loaded successfully!

Sample from training set:
  Text: tomo itinye coruet ne cepto iman
  Duration: 2.14s
  SNR: 45.9dB
  Audio data: 24453 bytes stored

✓ All preprocessing complete!


In [6]:
print("\n" + "="*70)
print("DATASET VERIFICATION".center(70))
print("="*70)

from datasets import load_from_disk
import soundfile as sf
import io

# Load dataset
loaded_dataset = load_from_disk('../processed_data/kalenjin_asr_dataset')

print("\n✓ Dataset loaded successfully!")

# Access raw data without automatic decoding
train_table = loaded_dataset['train']._data.table
sample_idx = 0

print(f"\nSample from training set:")
print(f"  Text: {train_table['text'][sample_idx].as_py()}")
print(f"  Duration: {train_table['duration'][sample_idx].as_py():.2f}s")
print(f"  SNR: {train_table['snr_db'][sample_idx].as_py():.1f}dB")

# Manually decode audio bytes with soundfile
audio_col = train_table['audio'][sample_idx]
audio_bytes = audio_col['bytes'].as_py()

if audio_bytes:
    audio_array, sr = sf.read(io.BytesIO(audio_bytes))
    print(f"  Audio shape: {audio_array.shape}")
    print(f"  Sample rate: {sr}Hz")
else:
    print(f"  Audio path: {audio_col['path'].as_py()}")

print("\n✓ All preprocessing complete!")



                         DATASET VERIFICATION                         

✓ Dataset loaded successfully!

Sample from training set:
  Text: tomo itinye coruet ne cepto iman
  Duration: 2.14s
  SNR: 45.9dB
  Audio shape: (130176,)
  Sample rate: 32000Hz

✓ All preprocessing complete!
